In [1]:
import project_path
from rsatools import *

ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
rng = prime_rng(8675_309)

p = rng.randprime(2**31, 2**32)
q = rng.randprime(2**31, 2**32)

n = p*q

lamb = carmichael(p, q)

e = 257
d = inverse257(lamb)

secrets = [p, q, n, lamb, e, d]

for num in secrets:
    bitlength = len(str(bin(num)))-2
    print(f"{bitlength:2.0f}    {num}")

32    3721615301
32    2961235187
64    11020578181798796287
59    423868391350613300
 9    257
58    222654602460438893


In [ ]:
m_plaintext = "Hello, W"

m_encoding = int(codecs.encode(m_plaintext, encoding="ascii_emoji").hex(), 16)

c_encoding = pow(m_encoding, e, mod=n)

c_plaintext = codecs.decode(bytes.fromhex(hex(c_encoding)[2:]), encoding="ascii_emoji")

r_encoding = pow(c_encoding, d, mod=n)

r_plaintext = codecs.decode(bytes.fromhex(hex(r_encoding)[2:]), encoding="ascii_emoji")

print(m_plaintext)
print(c_plaintext)
print(r_plaintext)

Hello, W
(a⭐lKhW3
Hello, W


fucking finally

# To Do:

- automatic padding

- break long messages into words (is this necessary?)

- make it distributable

- increase the bounds for randprime...

okay let's test the long message thing first

In [ ]:
p2 = rng.randprime(2**63, 2**64)
q2 = rng.randprime(2**63, 2**64)

n2 = p2*q2

lamb2 = carmichael(p2, q2)

e = 257
d2 = inverse257(lamb2)

secrets2 = [p2, q2, n2, lamb2, e, d2]

for num in secrets2:
    bitlength2 = len(str(bin(num)))-2
    print(f"{bitlength2:3.0f}    {num}")

 64    16735907832055708781
 64    14995498527984231137
128    250963281260071146001697860887864513997
123    5703710937728889681135601239268740320
  9    257
121    1908634788500717947773002749327282753


In [ ]:
m_plaintext2 = "Hello, World!"
m_encoding2 = int(codecs.encode(m_plaintext2, encoding="ascii_emoji").hex(), 16)
c_encoding2 = pow(m_encoding2, e, mod=n2)
r_encoding2 = pow(c_encoding2, d2, mod=n2)
r_plaintext2 = codecs.decode(bytes.fromhex(hex(r_encoding2)[2:]), encoding="ascii_emoji")

print(r_plaintext2)

c_plaintext2 = codecs.decode(bytes.fromhex(hex(c_encoding2)[2:]), encoding="ascii_emoji")

Hello, World!


KeyError: b'\xfd'

Okay, I thought we might have gotten past this bug, but I guess not.  I knew it was going to rear its ugly head at some point.  Now, codecs probably has some way of dealing with this exact sort of error- but statistically, about half of the stuff in the ciphertext will encounter this bug.  so dealing with it by putting in a blank space, or a ?, or something like that, will make the ciphertext really boring and ugly.  so instead I'm going to fake it.

In [2]:
_decode_emoji = {
    b'\x00': "\U0001F5FF",                                                         # null
    b'\x01': "\U0001f92f", b'\x02': "\U0001f60d", b'\x03': "\U0001f911",           # tc 1-3
    b'\x04': "\U0001F636", b'\x05': "\U0001f62d", b'\x06': "\U0001F642",           # tc 4-6
    b'\x07': "\U0001f514",                                                         # bell
    b'\x08': "\U0001F353", b'\x09': "\U0001F36C", b'\x0a': "\U0001F3B2",           # fe 0-2
    b'\x0b': "\u23F3", b'\x0c': "\U0001F48E", b'\x0d': "\U0001F4A9",               # fe 3-5
    b'\x0e': "\u270c", b'\x0f': "\u270b", b'\x10': "\U0001f978",                   # so, si, tc7
    b'\x11': "\u2600", b'\x12': "\u2601", b'\x13': "\u26a1", b'\x14': "\u2b50",    # dc 1-4
    b'\x15': "\U0001F641", b'\x16': "\U0001F634", b'\x17': "\U0001f9ca",           # tc 8-10
    b'\x18': "\U0001f383", b'\x19': "\U0001f384",                                  # cancel, end of medium
    b'\x1a': "\U0001f389", b'\x1b': "\U0001f37e",                                  # substitute, escape
    b'\x1c': "\u2660", b'\x1d': "\u2663", b'\x1e': "\u2665", b'\x1f': "\u2666",    # is 1-4
    b'\x7f': "\U0001F480",                                                         # delete
}
_decode_ascii = {num.to_bytes(): num.to_bytes().decode("ascii") for num in range(32, 127)}


In [7]:
{bytes.fromhex(hex(int(key.hex(), 16)+128)[2:]): _decode_emoji[key] for key in _decode_emoji}

{b'\x80': '🗿',
 b'\x81': '🤯',
 b'\x82': '😍',
 b'\x83': '🤑',
 b'\x84': '😶',
 b'\x85': '😭',
 b'\x86': '🙂',
 b'\x87': '🔔',
 b'\x88': '🍓',
 b'\x89': '🍬',
 b'\x8a': '🎲',
 b'\x8b': '⏳',
 b'\x8c': '💎',
 b'\x8d': '💩',
 b'\x8e': '✌',
 b'\x8f': '✋',
 b'\x90': '🥸',
 b'\x91': '☀',
 b'\x92': '☁',
 b'\x93': '⚡',
 b'\x94': '⭐',
 b'\x95': '🙁',
 b'\x96': '😴',
 b'\x97': '🧊',
 b'\x98': '🎃',
 b'\x99': '🎄',
 b'\x9a': '🎉',
 b'\x9b': '🍾',
 b'\x9c': '♠',
 b'\x9d': '♣',
 b'\x9e': '♥',
 b'\x9f': '♦',
 b'\xff': '💀'}

In [8]:
_decode_table = {
    **_decode_emoji, **_decode_ascii, 
    **{bytes.fromhex(hex(int(key.hex(), 16)+128)[2:]): _decode_emoji[key] for key in _decode_emoji}, 
    **{bytes.fromhex(hex(int(key.hex(), 16)+128)[2:]): _decode_ascii[key] for key in _decode_ascii}
}


In [9]:
print(_decode_table)

{b'\x00': '🗿', b'\x01': '🤯', b'\x02': '😍', b'\x03': '🤑', b'\x04': '😶', b'\x05': '😭', b'\x06': '🙂', b'\x07': '🔔', b'\x08': '🍓', b'\t': '🍬', b'\n': '🎲', b'\x0b': '⏳', b'\x0c': '💎', b'\r': '💩', b'\x0e': '✌', b'\x0f': '✋', b'\x10': '🥸', b'\x11': '☀', b'\x12': '☁', b'\x13': '⚡', b'\x14': '⭐', b'\x15': '🙁', b'\x16': '😴', b'\x17': '🧊', b'\x18': '🎃', b'\x19': '🎄', b'\x1a': '🎉', b'\x1b': '🍾', b'\x1c': '♠', b'\x1d': '♣', b'\x1e': '♥', b'\x1f': '♦', b'\x7f': '💀', b' ': ' ', b'!': '!', b'"': '"', b'#': '#', b'$': '$', b'%': '%', b'&': '&', b"'": "'", b'(': '(', b')': ')', b'*': '*', b'+': '+', b',': ',', b'-': '-', b'.': '.', b'/': '/', b'0': '0', b'1': '1', b'2': '2', b'3': '3', b'4': '4', b'5': '5', b'6': '6', b'7': '7', b'8': '8', b'9': '9', b':': ':', b';': ';', b'<': '<', b'=': '=', b'>': '>', b'?': '?', b'@': '@', b'A': 'A', b'B': 'B', b'C': 'C', b'D': 'D', b'E': 'E', b'F': 'F', b'G': 'G', b'H': 'H', b'I': 'I', b'J': 'J', b'K': 'K', b'L': 'L', b'M': 'M', b'N': 'N', b'O': 'O', b'P': 'P', b'Q'